In [23]:
using LowLevelFEM, LinearAlgebra

In [24]:
openGeometry("boxes.geo")

In [25]:
#openPreProcessor()

In [26]:
mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u);

In [27]:
bc_bottom = BoundaryCondition("bottom", ux=0, uy=0, uz=0)
bc_top = BoundaryCondition("top", ux=0, uz=0, uy=(x,y,z)->-x*(x-10) * z*(z-10) / 4250)

K = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U))
f = ∫(U ⋅ [0, 0, 0])

u = solveField(K, f, support=[bc_bottom, bc_top])

#showDoFResults(u, name="u", factor=1, visible=true)

nodal VectorField
[0.0; 0.0; … ; -0.03266022872370279; 0.0018497157399644643;;]


## Lagrange multiplier contact

The contact geometry is the same as in the penalty example, but no penalty
stiffness is introduced. The unilateral normal contact condition is enforced
with a Lagrange multiplier.

The full contact operator contains both local components,

$$
G:V_u\rightarrow V_c,\qquad
d=G(r+u)=(d_n,d_t,\ldots)^T.
$$

For frictionless contact only the active **normal** rows are constrained. Let

$$
P_n:V_c\rightarrow V_{n,a}
$$

select the normal component of the currently active contact nodes. Then

$$
G_n=P_nG,\qquad
E_n=EP_n^T,\qquad
B=E_nG_n.
$$

The KKT system is

$$
\begin{bmatrix}
K & B^T\\
B & 0
\end{bmatrix}
\begin{bmatrix}
\Delta u\\
\Delta\lambda
\end{bmatrix}
=
-
\begin{bmatrix}
K u-f+B^T\lambda\\
g_a
\end{bmatrix}.
$$

The active set is determined with a primal-dual criterion. The parameter
$\kappa$ used there is only an active-set scaling parameter; it is not a
penalty stiffness.


In [ ]:

using SparseArrays

r = nodePositionVector(U)

# The multiplier field has the same number of components as U.
# On the slave contact surface these components are interpreted in the
# local contact basis (n,t). For frictionless contact only the first
# component is used.
Λ = Field([mat], type=:VectorField, dim=3, fieldName=:λ)

println("contact")
@time L = contact(
    u,
    master="master",
    slave="slave",
    LagrangeMultiplierField=Λ,
    topology_tol=0.01
)

support = [bc_bottom, bc_top]
free_u = freeDoFs(U, support)

u_it = copy(u)
λ_it = vectorField(Λ, "body", [0, 0, 0])

# Zero multiplier block of the KKT system.
Zλ = SystemMatrix(spzeros(ndofs(Λ), ndofs(Λ)), Λ)

# PDAS scaling parameter; this is not a penalty stiffness.
κ = 1e7;



### Active normal selection

`L.Pa` selects complete local contact blocks, i.e. both $(n,t)$ components
of an active node. That is appropriate for a general frictional contact space,
but frictionless Lagrange contact must constrain only the normal component.

The small helper below therefore constructs a rectangular `SystemMatrix`
$P_n$ that selects one normal contact DoF per active node. All subsequent
operations remain ordinary LLFEM algebra.


In [29]:

function activeNormalSelection(
    C::Contact,
    active::AbstractVector{Bool}
)
    length(active) == length(C.slave_nodes) ||
        error("activeNormalSelection: incompatible active-set size.")

    pdim = C.U.pdim
    active_nodes = findall(active)
    na = length(active_nodes)

    rows = collect(1:na)
    cols = (active_nodes .- 1) .* pdim .+ 1
    vals = ones(Float64, na)

    P = sparse(rows, cols, vals, na, length(C.d))

    return SystemMatrix(
        P,
        nothing,
        nothing,
        nothing,
        nothing
    )
end


activeNormalSelection (generic function with 1 method)


The sign convention used below is

$$
g_n\ge 0 \quad\text{open/admissible},
\qquad
\lambda_n\le 0 \quad\text{compression}.
$$

The primal-dual active-set rule is therefore

$$
\lambda_n+\kappa g_n<0.
$$

Inactive normal multipliers are set to zero. Tangential multipliers are never
included among the free multiplier DoFs, so they remain zero automatically.


In [ ]:

# Normal multiplier DoFs corresponding to the contact candidate nodes.
pdim = L.U.pdim
normal_rows = 1:pdim:length(L.d)
λn_dofs = L.multiplier_dofs[normal_rows]

active_old = falses(length(L.slave_nodes))
active_old2 = falses(length(L.slave_nodes))

for iter in 1:40

    # ----------------------------------------------------------
    # Current contact geometry
    # ----------------------------------------------------------
    println("updateContact")
    @time updateContact!(L, u_it)

    λn = DoFs(λ_it)[λn_dofs]

    # ----------------------------------------------------------
    # Primal-dual active set
    #
    # g_n >= 0 : open/admissible
    # λ_n <= 0 : compression
    # ----------------------------------------------------------
    active = λn .+ κ .* L.gap_values .< 0.0

    # Detect an A -> B -> A active-set cycle.
    two_cycle =
        iter > 2 &&
        active == active_old2 &&
        active != active_old

    if two_cycle
        switching = active .!= active_old

        println(
            "Two-cycle detected: freezing ",
            count(switching),
            " switching contact points."
        )

        active[switching] .= active_old[switching]
    end

    # Inactive normal multipliers must vanish.
    DoFs(λ_it)[λn_dofs[.!active]] .= 0.0

    # ----------------------------------------------------------
    # Active normal contact algebra
    #
    # Pn : Vc -> Vna
    # Gn = Pn G
    # En = E Pn'
    # B  = En Gn
    # ----------------------------------------------------------
    println("activeNormalSelection")
    @time Pn = activeNormalSelection(L, active)

    println("Gn")
    @time Gn = Pn * L.G
    println("En")
    @time En = L.E * Pn'
    println("B")
    @time B  = En * Gn

    # Active normal gap embedded in the multiplier field.
    println("dna")
    @time dna = Pn * L.d
    println("gλ")
    @time gλ = En * dna

    # ----------------------------------------------------------
    # KKT residual
    #
    # r_u = K u - f + B' λ
    # r_λ = g_a
    # ----------------------------------------------------------
    println("r_u")
    @time r_u = K * u_it - f + B' * λ_it
    r_λ = gλ

    # ----------------------------------------------------------
    # KKT tangent
    #
    #     [ K   B' ]
    # A = [        ]
    #     [ B    0 ]
    # ----------------------------------------------------------
    println("A")
    @time A = SystemMatrix([
        K   B'
        B   Zλ
    ])

    println("res")
    @time res = SystemVector([r_u, r_λ])

    # Multiplier-field offset in the assembled multifield system.
    λoff = A.offsets[2]

    # The Newton correction contains only:
    #   - unconstrained displacement DoFs,
    #   - active NORMAL multiplier DoFs.
    free = vcat(
        free_u,
        λoff .+ λn_dofs[active]
    )

    Δx = zeros(Float64, size(A, 1))

    println("Δx")
    @time Δx[free] =
        -A[free, free] \ res.a[free, 1]

    Δu = @view Δx[1:λoff]
    Δλ = @view Δx[λoff+1:end]

    # Homogeneous displacement increment on prescribed displacement DoFs
    # follows automatically because those DoFs are not included in `free`.
    DoFs(u_it)[:] .+= Δu
    DoFs(λ_it)[:] .+= Δλ

    # ----------------------------------------------------------
    # Diagnostics
    # ----------------------------------------------------------
    Δactive = count(active .!= active_old)

    err_u =
        norm(Δu[free_u]) /
        max(norm(DoFs(u_it)[free_u]), eps())

    max_gap =
        any(active) ?
        maximum(abs.(L.gap_values[active])) :
        0.0

    λn = DoFs(λ_it)[λn_dofs]

    min_λ =
        any(active) ?
        minimum(λn[active]) :
        0.0

    max_λ =
        any(active) ?
        maximum(λn[active]) :
        0.0

    println(
        "iter = ", iter,
        ", active = ", count(active),
        ", Δactive = ", Δactive,
        ", max |gap| = ", max_gap,
        ", min λn = ", min_λ,
        ", max λn = ", max_λ,
        ", error = ", err_u
    )

    converged =
        Δactive == 0 &&
        max_gap < 1e-8 &&
        err_u < 1e-8

    active_old2 .= active_old
    active_old .= active

    converged && break
end

u_LM = u_it
λ_LM = λ_it

# Synchronize contact geometry with the final displacement.
updateContact!(L, u_LM);


In [ ]:

showDoFResults(u_LM, name="u LM", factor=1, visible=true)


0


## Contact gap and pressure

`L.d` is the reduced contact vector. Mapping it back to the displacement mesh
gives the ordinary LLFEM field representation, so the normal gap is simply the
first component.

With the sign convention used above the normal Lagrange multiplier is negative
in compression, therefore the positive contact pressure is

$$
p=-\lambda_n.
$$


In [ ]:

DD = VectorField(L.d)
gap = DD[1]

pressure = -λ_LM[1]

showElementResults(nodesToElements(pressure, onPhysicalGroup="slave"), name="p")
showElementResults(nodesToElements(gap, onPhysicalGroup="slave"), name="gap")


2

In [ ]:

# Optional postprocessing:
# showElementResults(
#     nodesToElements(pressure, onPhysicalGroup="slave"),
#     name="pressure",
#     visible=true
# )
#
# showElementResults(
#     nodesToElements(gap, onPhysicalGroup="slave"),
#     name="gap",
#     visible=true
# )

openPostProcessor()


XOpenIM() failed
Fontconfig warning: using without calling FcInit()
